In [301]:
import sys
!{sys.executable} -m pip install torch
import sys
!{sys.executable} -m pip install pennylane


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [302]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import math
import pennylane as qml
import matplotlib.pyplot as plt

In [67]:

layer_1 = torch.nn.Linear(6, 16)
layer_2 = torch.nn.Linear(16, 2)
softmax = torch.nn.Softmax(dim=1)

layers = [layer_1, layer_2, softmax]
model = torch.nn.Sequential(*layers)

In [ ]:
# все что связанно с датасетом
#class FinancialDataset(Dataset):
#    def __init__(self, file_path, window_size=4, target_col='Close', is_train=True, train_ratio=0.8):
#        self.window_size = window_size
#        df = pd.read_csv(file_path)
#        prices = df[target_col].values.reshape(-1, 1)
#        
#        log_returns = np.diff(np.log(prices), axis=0)
#        self.scaler = MinMaxScaler(feature_range=(0, np.pi))
#        
#        split_idx = int(len(log_returns) * train_ratio)
#        if is_train:
#            self.data = self.scaler.fit_transform(log_returns[:split_idx])
#        else:
#            self.scaler.fit(log_returns[:split_idx])
#            self.data = self.scaler.transform(log_returns[split_idx:])
#            
#        self.X, self.y = self._build_sequences()
#
#    def _build_sequences(self):
#        X, y = [], []
#        for i in range(len(self.data) - self.window_size):
#            X.append(self.data[i : i + self.window_size])
#            y.append(self.data[i + self.window_size])
#        return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)
#
#    def __len__(self): return len(self.X)
#    def __getitem__(self, idx): return self.X[idx], self.y[idx]
#

In [376]:
df = pd.read_csv('../data/ABC.csv')
df


,Date,Open,High,Low,Close,Adj Close,Volume
0,1995-04-04,2.937500,2.968750,2.812500,2.953125,2.273860,40387200
1,1995-04-05,2.906250,2.968750,2.843750,2.843750,2.189643,12236800
2,1995-04-06,2.843750,2.937500,2.843750,2.890625,2.225735,3776000
3,1995-04-07,2.906250,2.906250,2.843750,2.843750,2.189643,1920800
4,1995-04-10,2.843750,2.906250,2.843750,2.875000,2.213704,2047200
...,...,...,...,...,...,...,...
6288,2020-03-26,75.870003,85.180000,75.599998,84.160004,84.160004,3263800
6289,2020-03-27,80.940002,84.059998,80.180000,81.779999,81.779999,1600500
6290,2020-03-30,83.529999,88.550003,82.440002,88.099998,88.099998,1770200
6291,2020-03-31,88.239998,90.730003,87.110001,88.500000,88.500000,2074700


In [ ]:
#low плохо показал себя
#high тоже не очень
# Adj Close хуже всех
# Volume хорошо
# Open неплохо


In [465]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/ABC.csv')
feature_cols = ["Open", 'Volume']

# 1. Просто превращаем каждую строчку в % изменения относительно вчерашнего дня
# Метод .pct_change() встроен в Pandas и делает всё сам!
df_scaled = df[feature_cols].pct_change()

# 2. Считаем лог-доходность и таргет
df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
df['target'] = (df['log_returns'].shift(-1) > 0).astype(int)

# Склеиваем и чистим от первой пустой строки
df_scaled['target'] = df['target']
df_processed = df_scaled.dropna().reset_index(drop=True)

# 3. Нарезаем на 5-дневные окна
window_size = 64
X_list = []
y_list = []

features_array = df_processed[feature_cols].values
targets = df_processed['target'].values

for i in range(len(df_processed) - window_size):
    X_list.append(features_array[i : i + window_size])
    y_list.append(targets[i + window_size - 1])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)

y_hot = np.eye(2)[y].astype(np.float32)
X_train, X_test, y_hot_train, y_hot_test = train_test_split(X, y_hot, test_size=0.2, shuffle=False)

In [ ]:
X_train = torch.tensor(X_train, requires_grad=True).float()
y_hot_train = torch.tensor(y_hot_train).float()
X_test = torch.tensor(X_test, requires_grad=True).float()

batch_size = 64
batches = 200 // batch_size

data_loader_train = torch.utils.data.DataLoader(
    list(zip(X_train, y_hot_train)), batch_size=64, shuffle=True, drop_last=True
)
data_loader_test = torch.utils.data.DataLoader(
    list(zip(X_test, y_hot_test)), batch_size=64, shuffle=True, drop_last=True
)

In [388]:
y_hot_train

tensor([[0., 1.],
        [0., 1.],
        [0., 1.],
        ...,
        [0., 1.],
        [0., 1.],
        [1., 0.]])

In [ ]:
if __name__ == "__main__":
    FILE_PATH = '../data/ABC.csv'
    TOTAL_ROWS = 250     # Оптимальный размер
    WINDOW_SIZE = 4      
    BATCH_SIZE = 32      
    EPOCHS = 50          
    LEARNING_RATE = 0.015
    
    print("⏳ Шаг 1. Загрузка и Z-score предобработка...")
    train_dataset = FinancialDataset(FILE_PATH, window_size=WINDOW_SIZE, is_train=True, max_rows=TOTAL_ROWS)
    test_dataset = FinancialDataset(FILE_PATH, window_size=WINDOW_SIZE, is_train=False, max_rows=TOTAL_ROWS)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    
    #trainer = ComparativeTrainer(train_loader, test_loader, train_dataset.mean, train_dataset.std)

In [ ]:
class ComparativeTrainer:
    def __init__(self, train_loader, test_loader, dataset_mean, dataset_std):
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.mean = dataset_mean
        self.std = dataset_std
        self.criterion = nn.MSELoss()

    def train_model(self, model, epochs=40, lr=0.01, name="Модель"):
        optimizer = optim.Adam(model.parameters(), lr=lr)
        print(f"\n🚀 Обучение: {name}...")
        
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            for inputs, targets in self.train_loader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = self.criterion(outputs, targets)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                epoch_loss += loss.item()
            
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"  Эпоха [{epoch+1}/{epochs}] -> Лосс: {epoch_loss/len(self.train_loader):.6f}")

    def evaluate_and_compare(self, hybrid_model, classic_model, plot_days=50):
        hybrid_model.eval()
        classic_model.eval()
        
        actuals, hybrid_preds, classic_preds = [], [], []
        
        with torch.no_grad():
            for inputs, targets in self.test_loader:
                actuals.extend(targets.numpy().flatten())
                hybrid_preds.extend(hybrid_model(inputs).numpy().flatten())
                classic_preds.extend(classic_model(inputs).numpy().flatten())
                
        # Денормализация через среднее и std (возвращаем к исходным лог-доходностям)
        real_actuals = np.array(actuals) * self.std + self.mean
        real_hybrid = np.array(hybrid_preds) * self.std + self.mean
        real_classic = np.array(classic_preds) * self.std + self.mean
        
        # Построение графиков
        plt.figure(figsize=(11, 4.5))
        view_days = min(plot_days, len(real_actuals))
        
        days = np.arange(1, view_days + 1)
        plt.plot(days, real_actuals[-view_days:], label="Рынок (Факт)", color="blue", marker='o', alpha=0.5)
        plt.plot(days, real_hybrid[-view_days:], label="Квантовый Гибрид", color="red", linestyle="--", marker='x')
        plt.plot(days, real_classic[-view_days:], label="Чистая Классика", color="orange", linestyle=":", marker='s')
        
        plt.title("Сравнение моделей на реальных данных лог-доходности")
        plt.xlabel("Торговые дни")
        plt.ylabel("Лог-доходность")
        plt.legend()
        plt.grid(True)
        plt.show()

In [457]:
import pennylane as qml

n_qubits = 8
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs * torch.pi, wires=range(n_qubits), rotation="X") #переводит кубит в нужное состояние
    #qml.BasicEntanglerLayers(weights, wires=range(n_qubits)) #запутывает кубиты между собой
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)] #выдает число, определяющее мат ожидание класса с помощью оператора PauliZ

In [419]:
n_layers = 1
weight_shapes = {"weights": (n_layers, n_qubits)}

In [484]:
class HybridModel(torch.nn.Module):
    def __init__(self):
        super().__init__() # Замените YourModelName на имя вашего класса
        
        # 1. Первый слой принимает входной размер (после flatten) и выдает 32 признака
        self.clayer_1 = nn.Linear(128, 32)
        
        self.dropout = nn.Dropout(p=0.5) # Вероятность дропаута можно настроить
        
        # 2. Слой для второй ветки. Получает 16 признаков и должен вернуть тоже 16,
        # чтобы после склеивания (16 + 16) снова получилось 32 для clayer_3.
        self.qlayer_2 = nn.Linear(16, 16) 
        
        # 3. Принимает склеенный тензор (16 от x_1 + 16 от x_2 = 32)
        self.clayer_3 = nn.Linear(32, 64) # Количество выходов (например, 64) можно менять
        
        # 4. Финальный слой, выдающий итоговое количество классов/значений
        self.clayer_2 = nn.Linear(64, 2)
        
        

    def forward(self, x): # тензор проходит через нашу схему
        x = torch.flatten(x, start_dim=1)
        x = self.clayer_1(x)
        x = torch.relu(x) # Активация, чтобы модель могла гнуть линии тренда
        x = self.dropout(x)
        
        # Разделяем тензор посередине на две части по размерности признаков (dim=1)
        x_1, x_2 = torch.split(x, 16, dim=1)
        
        # Пропускаем вторую ветку через слой qlayer_2 (первая x_1 остается неизменной)
        x_2 = self.qlayer_2(x_2)
        
        # Соединяем ветки обратно в один тензор по той же оси
        x = torch.cat([x_1, x_2], dim=1)
        
        x = self.clayer_3(x)
        x = self.clayer_2(x)
        return x

model = HybridModel()

In [468]:

loss = torch.nn.L1Loss() # средний квадрат

In [485]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003) 
criterion = torch.nn.CrossEntropyLoss()

epochs = 40  # Дай ей чуть больше времени раскачаться со стабильным батчем
batch_size_real = len(data_loader_train)

criterion = torch.nn.CrossEntropyLoss()
#print("Веса квантового слоя ДО:", model.qlayer_1.weights)
for epoch in range(epochs):
    running_loss = 0
    for batch_X, batch_y in data_loader_train:
        optimizer.zero_grad()  # Обнуляем прошлые градиенты
        
        # Пропускаем батч через модель. 
        # На выходе получаем тензор вероятностей размера [5, 2]
        outputs = model(batch_X) 
        
        # Считаем ошибку BCELoss между предсказанием [5, 2] и реальным batch_y [5, 2]
        loss_evaluated = criterion(outputs, batch_y)
        
        # Делаем магию PyTorch: считаем градиенты и делаем шаг обучения
        loss_evaluated.backward()
        optimizer.step()
    
        running_loss += loss_evaluated.item()

    y_pred_test = model(X_test)
    predictions = torch.argmax(y_pred_test, axis=1).detach().numpy()
    true_labels = torch.argmax(torch.tensor(y_hot_test), dim=1).numpy()
    correct = [1 if p == p_true else 0 for p, p_true in zip(predictions, true_labels)]
    accuracy = sum(correct) / len(correct)
    print(f"Accuracy: {accuracy * 100}%")
    
    avg_loss = running_loss / len(data_loader_train)
    print("Average loss over epoch {}: {:.4f}".format(epoch + 1, avg_loss))
    #print("Веса квантового слоя ПОСЛЕ:", model.qlayer_1.weights)

y_pred_test = model(X_test)
predictions = torch.argmax(y_pred_test, axis=1).detach().numpy()
true_labels = torch.argmax(torch.tensor(y_hot_test), dim=1).numpy()
correct = [1 if p == p_true else 0 for p, p_true in zip(predictions, true_labels)]
accuracy = sum(correct) / len(correct)
print(f"Accuracy: {accuracy * 100}%")

Accuracy: 51.20385232744783%
Average loss over epoch 1: 0.7724
Accuracy: 51.28410914927769%
Average loss over epoch 2: 0.7455
Accuracy: 51.36436597110754%
Average loss over epoch 3: 0.7108
Accuracy: 51.84590690208668%
Average loss over epoch 4: 0.7055
Accuracy: 51.28410914927769%
Average loss over epoch 5: 0.7015
Accuracy: 51.84590690208668%
Average loss over epoch 6: 0.7002
Accuracy: 51.52487961476726%
Average loss over epoch 7: 0.6927
Accuracy: 51.52487961476726%
Average loss over epoch 8: 0.6912
Accuracy: 51.92616372391653%
Average loss over epoch 9: 0.6928
Accuracy: 50.0%
Average loss over epoch 10: 0.6910
Accuracy: 51.20385232744783%
Average loss over epoch 11: 0.6915
Accuracy: 51.4446227929374%
Average loss over epoch 12: 0.6869
Accuracy: 51.04333868378812%
Average loss over epoch 13: 0.6805
Accuracy: 52.80898876404494%
Average loss over epoch 14: 0.6817
Accuracy: 51.52487961476726%
Average loss over epoch 15: 0.6825
Accuracy: 50.32102728731942%
Average loss over epoch 16: 0.6832

In [265]:
import torch
import pennylane as qml

n_qubits = 4
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs * torch.pi, wires=range(n_qubits), rotation="X") 
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits)) 
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

n_layers = 1
weight_shapes = {"weights": (n_layers, n_qubits)}

class HybridModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # ИСПРАВЛЕНО: 16 дней * 6 фичей = 96 входов
        self.clayer_1 = torch.nn.Linear(192, 4) 
        self.qlayer_1 = qml.qnn.TorchLayer(qnode, weight_shapes)
        self.clayer_2 = torch.nn.Linear(4, 2)
        # ИСПРАВЛЕНО: Удалили Softmax!

    def forward(self, x):
        # ИНСАЙТ ИЗ СТАТЬИ 2 (NLinear): Вычитаем последнее значение окна для стационарности
        # Берем значения последнего дня в окне (x[:, -1, :]) и вычитаем из всего батча
        seq_last = x[:, -1:, :].detach()
        x = x - seq_last 

        x = torch.flatten(x, start_dim=1)
        x = self.clayer_1(x)
        x = self.qlayer_1(x)
        x = self.clayer_2(x)
        return x # Возвращаем сырые логиты для CrossEntropyLoss

model = HybridModel()

# ИСПРАВЛЕНО: Adam вместо SGD, иначе кванты не обучатся
optimizer = torch.optim.Adam(model.parameters(), lr=0.005) 
criterion = torch.nn.CrossEntropyLoss() 

epochs = 30
batch_size_real = len(data_loader_train)

print("Веса ДО:", model.qlayer_1.weights)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch_X, batch_y in data_loader_train:
        optimizer.zero_grad()  
        
        outputs = model(batch_X) 
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
    
        running_loss += loss.item()
    
    avg_loss = running_loss / batch_size_real
    print(f"Epoch {epoch + 1}/{epochs} -> Average Loss: {avg_loss:.4f}")

print("Веса ПОСЛЕ:", model.qlayer_1.weights)

Веса ДО: Parameter containing:
tensor([[3.8451, 5.5557, 2.0280, 1.7467]], requires_grad=True)


KeyboardInterrupt: 

In [ ]:
# 1. ИСПРАВЛЯЕМ DATALOADER: ставим честный батч 32, чтобы усреднить шум
data_loader_train = torch.utils.data.DataLoader(
    list(zip(X_train, y_hot_train)), batch_size=32, shuffle=True, drop_last=True
)
data_loader_test = torch.utils.data.DataLoader(
    list(zip(X_test, y_hot_test)), batch_size=32, shuffle=False, drop_last=True
)

# 2. ДОБАВЛЯЕМ КЛАССИЧЕСКУЮ СТАБИЛИЗАЦИЮ В МОДЕЛЬ
class HybridModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # 16 дней * 6 колонок = 96 входов. Сжимаем до 16 нейронов
        self.clayer_1 = torch.nn.Linear(96, 16)
        # Dropout на 20% временно отключает случайные связи, заставляя модель искать реальные паттерны
        self.dropout = torch.nn.Dropout(p=0.2) 
        # Сжимаем до 4 нейронов строго под 4 кубита
        self.clayer_to_quantum = torch.nn.Linear(16, 4)
        
        self.qlayer_1 = qml.qnn.TorchLayer(qnode, weight_shapes)
        self.clayer_2 = torch.nn.Linear(4, 2)
        # Убрали Softmax навсегда!

    def forward(self, x): 
        x = torch.flatten(x, start_dim=1)
        x = self.clayer_1(x)
        x = torch.relu(x) # Активация, чтобы модель могла гнуть линии тренда
        x = self.dropout(x)
        
        x = self.clayer_to_quantum(x)
        x = torch.tanh(x) # Зажимаем в (-1, 1) для квантовых гейтов
        
        x = self.qlayer_1(x)
        x = self.clayer_2(x)
        return x

model = HybridModel()

# 3. ИСПРАВЛЯЕМ ОПТИМИЗАТОР: Adam вместо SGD
optimizer = torch.optim.Adam(model.parameters(), lr=0.003) 
criterion = torch.nn.CrossEntropyLoss()

epochs = 40  # Дай ей чують больше времени раскачаться со стабильным батчем
batch_size_real = len(data_loader_train)

for epoch in range(epochs):
    running_loss = 0
    for batch_X, batch_y in data_loader_train:
        optimizer.zero_grad()  
        
        outputs = model(batch_X) 
        loss_evaluated = criterion(outputs, batch_y)
        
        loss_evaluated.backward()
        optimizer.step()
    
        running_loss += loss_evaluated.item()
    
    avg_loss = running_loss / batch_size_real
    print("Average loss over epoch {}: {:.4f}".format(epoch + 1, avg_loss))
    y_pred_test = model(X_test)
    predictions = torch.argmax(y_pred_test, axis=1).detach().numpy()
    true_labels = torch.argmax(torch.tensor(y_hot_test), dim=1).numpy()
    correct = [1 if p == p_true else 0 for p, p_true in zip(predictions, true_labels)]
    accuracy = sum(correct) / len(correct)
    print(f"Accuracy: {accuracy * 100}%")

Average loss over epoch 1: 0.6953


TypeError: flatten(): argument 'input' (position 1) must be Tensor, not numpy.ndarray

классика


In [522]:
# --- 2. Чисто классическая архитектура (ДВОЙНИК) ---
class ClassicalModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Вход: 64 окна * 2 фичи = 128
        self.clayer_1 = nn.Linear(128, 16) 
        self.dropout = nn.Dropout(p=0.5)
        
        # Ветка 1: Классическая
        self.branch_1 = nn.Linear(8, 8)
        
        # Ветка 2: Тоже классическая (заменили квантовую)
        self.branch_2 = nn.Linear(8, 8)
        
        # Слияние: 8 + 8 = 16
        self.clayer_out = nn.Linear(16, 2)
        
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = torch.relu(self.clayer_1(x))
        x = self.dropout(x)
        
        # Разделяем 16 признаков на два тензора по 8
        x_1, x_2 = torch.split(x, 8, dim=1)
        
        # Пропускаем через классические слои
        x_1 = torch.relu(self.branch_1(x_1))
        x_2 = torch.relu(self.branch_2(x_2)) # Здесь раньше был квантовый слой
        
        x_concat = torch.cat([x_1, x_2], dim=1)
        out = self.clayer_out(x_concat)
        return out

In [527]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split

# --- 1. Подготовка данных ---
df = pd.read_csv('../data/ABC.csv')
feature_cols = ["Open", 'Volume']

df_scaled = df[feature_cols].pct_change()
df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
df['target'] = (df['log_returns'].shift(-1) > 0).astype(int)

df_scaled['target'] = df['target']
df_processed = df_scaled.dropna().reset_index(drop=True)

window_size = 64
X_list, y_list = [], []
features_array = df_processed[feature_cols].values
targets = df_processed['target'].values

for i in range(len(df_processed) - window_size):
    X_list.append(features_array[i : i + window_size])
    y_list.append(targets[i + window_size - 1])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
y_hot = np.eye(2)[y].astype(np.float32)

X_train, X_test, y_hot_train, y_hot_test = train_test_split(X, y_hot, test_size=0.2, shuffle=False)

X_train = torch.tensor(X_train).float()
y_hot_train = torch.tensor(y_hot_train).float()
X_test = torch.tensor(X_test).float()
y_hot_test = torch.tensor(y_hot_test).float()

data_loader_train = torch.utils.data.DataLoader(list(zip(X_train, y_hot_train)), batch_size=64, shuffle=True, drop_last=True)
data_loader_test = torch.utils.data.DataLoader(list(zip(X_test, y_hot_test)), batch_size=64, shuffle=False, drop_last=True)

# --- 2. Квантовая часть ---
n_qubits = 8
dev = qml.device("default.qubit", wires=n_qubits) # default.qubit работает стабильнее в гибридных схемах

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits)) # Кодируем данные в углы
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits)) # Квантовая запутанность
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

n_layers = 2 # Количество слоев квантовой сети
weight_shapes = {"weights": (n_layers, n_qubits)}
qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)


# --- 4. Обучение ---
model = HybridModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss()
epochs = 30

for epoch in range(epochs):
    model.train()
    running_loss = 0
    for batch_X, batch_y in data_loader_train:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Оценка на тесте
    # Тестирование
    
    with torch.no_grad():
        outputs_test = model(X_test)
        predictions = (torch.sigmoid(outputs_test) >= 0.5).float()
        accuracy = (predictions == y_hot_test).float().mean().item()
        
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        
    print(f"Эпоха {epoch+1:02d}/{epochs} | Трейн Лосс: {running_loss/len(data_loader_train):.4f} | Тест Аккураси: {accuracy * 100:.2f}%")

print(f"\n[Итог] Лучшая точность на тесте: {best_accuracy * 100:.2f}%")

Эпоха 01/30 | Трейн Лосс: 0.7246 | Тест Аккураси: 48.15%
Эпоха 02/30 | Трейн Лосс: 0.7091 | Тест Аккураси: 49.04%
Эпоха 03/30 | Трейн Лосс: 0.7026 | Тест Аккураси: 48.03%
Эпоха 04/30 | Трейн Лосс: 0.6995 | Тест Аккураси: 49.00%
Эпоха 05/30 | Трейн Лосс: 0.6970 | Тест Аккураси: 49.24%
Эпоха 06/30 | Трейн Лосс: 0.6954 | Тест Аккураси: 50.12%
Эпоха 07/30 | Трейн Лосс: 0.6948 | Тест Аккураси: 50.60%
Эпоха 08/30 | Трейн Лосс: 0.6927 | Тест Аккураси: 50.64%
Эпоха 09/30 | Трейн Лосс: 0.6906 | Тест Аккураси: 50.16%
Эпоха 10/30 | Трейн Лосс: 0.6897 | Тест Аккураси: 51.57%
Эпоха 11/30 | Трейн Лосс: 0.6885 | Тест Аккураси: 51.04%
Эпоха 12/30 | Трейн Лосс: 0.6895 | Тест Аккураси: 51.93%
Эпоха 13/30 | Трейн Лосс: 0.6890 | Тест Аккураси: 49.52%
Эпоха 14/30 | Трейн Лосс: 0.6895 | Тест Аккураси: 51.89%
Эпоха 15/30 | Трейн Лосс: 0.6883 | Тест Аккураси: 50.80%
Эпоха 16/30 | Трейн Лосс: 0.6886 | Тест Аккураси: 52.41%
Эпоха 17/30 | Трейн Лосс: 0.6876 | Тест Аккураси: 50.56%
Эпоха 18/30 | Трейн Лосс: 0.689

кванты

In [526]:
# --- 3. Гибридная архитектура ---
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Вход: 64 окна * 2 фичи = 128
        self.clayer_1 = nn.Linear(128, 16) 
        self.dropout = nn.Dropout(p=0.5)
        
        # Ветка 1: Классическая
        self.branch_classic = nn.Linear(8, 8)
        
        # Ветка 2: Квантовая
        self.branch_quantum = qlayer
        
        # Слияние: 8 (от классики) + 8 (от квантов) = 16
        self.clayer_out = nn.Linear(16, 2)
        
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = torch.relu(self.clayer_1(x))
        x = self.dropout(x)
        
        # Разделяем 16 признаков на два тензора по 8
        x_class, x_quant = torch.split(x, 8, dim=1)
        
        # Масштабируем данные для квантового слоя (углы от 0 до pi)
        x_quant = torch.sigmoid(x_quant) * torch.pi 
        
        x_class = torch.relu(self.branch_classic(x_class))
        x_quant = self.branch_quantum(x_quant)
        
        x_concat = torch.cat([x_class, x_quant], dim=1)
        out = self.clayer_out(x_concat)
        return out


In [513]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split

# --- 1. Подготовка данных (Окно 32, две фичи Open и Volume) ---
df = pd.read_csv('../data/ABC.csv')
feature_cols = ["Open", 'Volume']

df_scaled = df[feature_cols].pct_change()
df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
df['target'] = (df['log_returns'].shift(-1) > 0).astype(np.float32)

df_scaled['target'] = df['target']
df_processed = df_scaled.dropna().reset_index(drop=True)

window_size = 32 
X_list, y_list = [], []
features_array = df_processed[feature_cols].values
targets = df_processed['target'].values

for i in range(len(df_processed) - window_size):
    X_list.append(features_array[i : i + window_size])
    y_list.append(targets[i + window_size - 1])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).float()
X_test = torch.tensor(X_test).float()
y_test = torch.tensor(y_test).float()

data_loader_train = torch.utils.data.DataLoader(list(zip(X_train, y_train)), batch_size=32, shuffle=True, drop_last=True)

# --- 2. Квантовая часть (4 кубита) ---
n_qubits = 4 
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="X")
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

n_layers = 2
weight_shapes = {"weights": (n_layers, n_qubits)}
qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

# --- 3. Исправленная Архитектура Модели ---
class RobustHybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Вход: 32 окна * 2 фичи = 64. На выходе делаем 8 признаков.
        self.clayer_1 = nn.Linear(64, 8) 
        self.dropout = nn.Dropout(p=0.5)
        
        # Разделяем 8 признаков пополам: 4 идет в классику, 4 в кванты
        self.branch_classic = nn.Linear(4, 4)
        self.branch_quantum = qlayer # Принимает 4 признака и возвращает 4
        
        # Слияние: 4 (классика) + 4 (кванты) = 8 -> выходной бинарный нейрон
        self.clayer_out = nn.Linear(8, 1)
        
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = torch.relu(self.clayer_1(x))
        x = self.dropout(x)
        
        # split(x, 4) делит тензор из 8 элементов на части по 4 элемента!
        x_class, x_quant = torch.split(x, 4, dim=1)
        
        # Масштабируем данные под кванты [0, pi]
        x_quant = torch.sigmoid(x_quant) * torch.pi 
        
        x_class = torch.relu(self.branch_classic(x_class))
        x_quant = self.branch_quantum(x_quant) # Теперь сюда прилетает ровно 4 фичи
        
        # Склеиваем обратно в 8
        x_concat = torch.cat([x_class, x_quant], dim=1)
        out = self.clayer_out(x_concat)
        return out

# --- 4. Настройка обучения ---
model = RobustHybridModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss() 

epochs = 40
best_accuracy = 0.0

print("Запуск исправленной модели...")
for epoch in range(epochs):
    model.train()
    running_loss = 0
    for batch_X, batch_y in data_loader_train:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    


Запуск исправленной модели...
